In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

files_info = {
    "metric": {"filename": "metric.csv", "name_col": "kpi_name"},
    "trace": {"filename": "trace.csv", "name_col": "trace_name"},
    "log": {"filename": "log.csv", "name_col": "log_name"},
    "error_logs": {"filename": "error_logs.csv", "name_col": "message"},
}

def ts_to_local_iso(ts):
    # handle NaN
    if pd.isna(ts):
        return None
    # ts expected as seconds since epoch (int/float)
    try:
        # create timezone-aware datetime in UTC then convert
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    except Exception:
        return str(ts)

summary = {}

for key, info in files_info.items():
    fn = info["filename"]
    name_col = info["name_col"]
    try:
        df = pd.read_csv(fn)
    except Exception as e:
        # if file missing or unreadable, store error info
        summary[key] = {"error": f"failed to read {fn}: {e}"}
        continue

    total_rows = int(df.shape[0])

    # timestamps
    if 'timestamp' in df.columns and total_rows > 0:
        min_ts = ts_to_local_iso(df['timestamp'].min())
        max_ts = ts_to_local_iso(df['timestamp'].max())
    else:
        min_ts = None
        max_ts = None

    # cmdb_id unique list (up to 20)
    if 'cmdb_id' in df.columns:
        unique_cmdb = pd.Series(df['cmdb_id'].dropna().unique()).astype(str)
        unique_cmdb_list = unique_cmdb.tolist()[:20]
        num_unique_cmdb = int(unique_cmdb.nunique())
    else:
        unique_cmdb_list = []
        num_unique_cmdb = 0

    # name column unique list (up to 50)
    if name_col in df.columns:
        unique_names = pd.Series(df[name_col].dropna().unique()).astype(str)
        unique_names_list = unique_names.tolist()[:50]
        num_unique_names = int(unique_names.nunique())
        # top 10 most frequent values with counts
        top_vals_series = df[name_col].astype(str).value_counts().head(10)
        top_values = [{"value": v, "count": int(c)} for v, c in zip(top_vals_series.index.tolist(), top_vals_series.values.tolist())]
    else:
        unique_names_list = []
        num_unique_names = 0
        top_values = []

    # sample first 5 rows, convert timestamp to local ISO if present, keep only first 5 rows
    sample_df = df.head(5).copy()
    if 'timestamp' in sample_df.columns:
        sample_df['timestamp'] = sample_df['timestamp'].apply(ts_to_local_iso)
    # convert to records (compact)
    sample_records = sample_df.to_dict(orient='records')

    # assemble compact summary for this file
    summary[key] = {
        "filename": fn,
        "total_rows": total_rows,
        "min_timestamp": min_ts,
        "max_timestamp": max_ts,
        "num_unique_cmdb_id": num_unique_cmdb,
        "sample_cmdb_id_list_up_to_20": unique_cmdb_list,
        f"num_unique_{name_col}": num_unique_names,
        f"sample_{name_col}_list_up_to_50": unique_names_list,
        f"top_10_most_frequent_{name_col}_values": top_values,
        "sample_first_5_rows": sample_records,
    }

summary

```
Out[1]:
```
```python
summary_text = (
    "Summary of telemetry files:\n\n"
    "metric.csv: 327,084 rows. Time range 2022-03-21 21:30 to 2022-03-21 22:30 (UTC+8). "
    "59 unique cmdb_id (sample includes adservice, adservice-0, adservice-1, ...). "
    "1,436 unique KPI names. Top KPI names by frequency include several container.node-6 network metrics (e.g., container.node-6.container_network_receive_MB.eth0) and container.node-6.socket/tasks metrics. "
    "Sample rows show adservice KPIs (e.g., app.grpc.count = 144.0).\n\n"
    "trace.csv: 22,376 rows. Time range 2022-03-21 22:00 to 2022-03-21 22:29 (UTC+8). "
    "39 unique cmdb_id (sample includes adservice-0, adservice-1, adservice-2, ...). "
    "196 unique trace names. Top trace names are traces from frontend pods (e.g., trace.from_frontend-2.* and trace.from_frontend-0.*). "
    "Sample rows show per-pod trace features such as trace.from_frontend-0.duration_mean and row_count.\n\n"
    "log.csv: 1,562 rows. Time range 2022-03-21 22:00 to 2022-03-21 22:30 (UTC+8). "
    "31 unique cmdb_id (sample includes adservice-0, cartservice-0, frontend-0, ...). "
    "2 unique log names (log.error_count, log.row_count), each appearing 781 times. "
    "Sample rows show log.error_count = 0.0 for the shown entries.\n\n"
    "error_logs.csv: 0 rows (no error log messages present).\n\n"
    "Overall: metric contains the largest volume and many KPI types (network metrics from node-6 stand out in frequency). Trace and log files cover a shorter time window and fewer components; error_logs is empty."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'filename': 'metric.csv', 'total_rows': 327084, 'min_timestamp': '2022-03-21 21:30:00+0800', 'max_timestamp': '2022-03-21 22:30:00+0800', 'num_unique_cmdb_id': 59, 'sample_cmdb_id_list_up_to_20': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2'], 'num_unique_kpi_name': 1436, 'sample_kpi_name_list_up_to_50': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.MarkSweepCompact'], 'top_10_most_frequent_kpi_name_values': [{'value': 'container.node-6.container_network_receive_MB.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_receive_errors.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_receive_packets.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_receive_packets_dropped.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_transmit_MB.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_transmit_errors.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_transmit_packets.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_transmit_packets_dropped.eth0', 'count': 2196}, {'value': 'container.node-6.container_sockets', 'count': 2074}, {'value': 'container.node-6.container_tasks_state.uninterruptible', 'count': 2074}], 'sample_first_5_rows': [{'timestamp': '2022-03-21 21:30:00+0800', 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.count', 'value': 144.0}, {'timestamp': '2022-03-21 21:30:00+0800', 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.mrt', 'value': 0.0036208333333326}, {'timestamp': '2022-03-21 21:30:00+0800', 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.rr', 'value': 100.0}, {'timestamp': '2022-03-21 21:30:00+0800', 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.sr', 'value': 100.0}, {'timestamp': '2022-03-21 21:30:00+0800', 'cmdb_id': 'adservice', 'kpi_name': 'app.http.count', 'value': 24.0}]}, 'trace': {'filename': 'trace.csv', 'total_rows': 22376, 'min_timestamp': '2022-03-21 22:00:00+0800', 'max_timestamp': '2022-03-21 22:29:00+0800', 'num_unique_cmdb_id': 39, 'sample_cmdb_id_list_up_to_20': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend-2'], 'num_unique_trace_name': 196, 'sample_trace_name_list_up_to_50': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.error_rate', 'trace.self.row_count', 'trace.from_checkoutservice2-0.duration_mean', 'trace.from_checkoutservice2-0.duration_p95', 'trace.from_checkoutservice2-0.error_rate', 'trace.from_checkoutservice2-0.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.to_cartservice-0.duration_mean', 'trace.to_cartservice-0.duration_p95', 'trace.to_cartservice-0.error_rate', 'trace.to_cartservice-0.row_count', 'trace.to_cartservice-1.duration_mean', 'trace.to_cartservice-1.duration_p95', 'trace.to_cartservice-1.error_rate', 'trace.to_cartservice-1.row_count', 'trace.to_currencyservice-0.duration_mean', 'trace.to_currencyservice-0.duration_p95', 'trace.to_currencyservice-0.error_rate', 'trace.to_currencyservice-0.row_count', 'trace.to_currencyservice-1.duration_mean', 'trace.to_currencyservice-1.duration_p95', 'trace.to_currencyservice-1.error_rate', 'trace.to_currencyservice-1.row_count', 'trace.to_currencyservice-2.duration_mean', 'trace.to_currencyservice-2.duration_p95'], 'top_10_most_frequent_trace_name_values': [{'value': 'trace.from_frontend-2.row_count', 'count': 562}, {'value': 'trace.from_frontend-2.error_rate', 'count': 562}, {'value': 'trace.from_frontend-2.duration_p95', 'count': 562}, {'value': 'trace.from_frontend-2.duration_mean', 'count': 562}, {'value': 'trace.from_frontend-0.row_count', 'count': 556}, {'value': 'trace.from_frontend-0.error_rate', 'count': 556}, {'value': 'trace.from_frontend-0.duration_p95', 'count': 556}, {'value': 'trace.from_frontend-0.duration_mean', 'count': 556}, {'value': 'trace.self.row_count', 'count': 420}, {'value': 'trace.self.error_rate', 'count': 420}], 'sample_first_5_rows': [{'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.duration_mean', 'value': 1.8e-05}, {'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.duration_p95', 'value': 2.3e-05}, {'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.error_rate', 'value': 0.0}, {'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.row_count', 'value': 6.0}, {'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-2.duration_mean', 'value': 2.9e-05}]}, 'log': {'filename': 'log.csv', 'total_rows': 1562, 'min_timestamp': '2022-03-21 22:00:00+0800', 'max_timestamp': '2022-03-21 22:30:00+0800', 'num_unique_cmdb_id': 31, 'sample_cmdb_id_list_up_to_20': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-2', 'paymentservice-0', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2'], 'num_unique_log_name': 2, 'sample_log_name_list_up_to_50': ['log.error_count', 'log.row_count'], 'top_10_most_frequent_log_name_values': [{'value': 'log.error_count', 'count': 781}, {'value': 'log.row_count', 'count': 781}], 'sample_first_5_rows': [{'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-0', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-0', 'log_name': 'log.row_count', 'value': 32.0}, {'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-1', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-1', 'log_name': 'log.row_count', 'value': 26.0}, {'timestamp': '2022-03-21 22:00:00+0800', 'cmdb_id': 'adservice-2', 'log_name': 'log.error_count', 'value': 0.0}]}, 'error_logs': {'filename': 'error_logs.csv', 'total_rows': 0, 'min_timestamp': None, 'max_timestamp': None, 'num_unique_cmdb_id': 0, 'sample_cmdb_id_list_up_to_20': [], 'num_unique_message': 0, 'sample_message_list_up_to_50': [], 'top_10_most_frequent_message_values': [], 'sample_first_5_rows': []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse variable names if present
tz = pytz.timezone('Asia/Shanghai')

# read metric file
metric_df = pd.read_csv("metric.csv")

# convert timestamp to timezone-aware datetime in Asia/Shanghai
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) compute global P95 and P5 per (cmdb_id, kpi_name) using entire series
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(P95=lambda x: float(x.quantile(0.95)), P5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) filter to incident time window (inclusive) in UTC+8
start = pd.Timestamp('2022-03-21 22:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
window_df = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] <= end)].copy()

# merge thresholds into window rows so we can compute flags using global thresholds
window_df = window_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# flags for anomalies based on global thresholds (rule 9)
window_df['anom_high'] = (window_df['value'] >= window_df['P95']).astype(int)
window_df['anom_low'] = (window_df['value'] <= window_df['P5']).astype(int)

# prepare anom timestamp column (ts only when anomaly)
window_df['anom_ts'] = window_df['ts'].where((window_df['anom_high'] == 1) | (window_df['anom_low'] == 1))

# aggregate per pair within the window
window_agg = (
    window_df
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count_high=('anom_high', 'sum'),
        anomaly_count_low=('anom_low', 'sum'),
        earliest_anomaly_timestamp_in_window=('anom_ts', 'min'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
    )
    .reset_index()
)

# 2b) Include pairs that have zero points in window: left-join thresholds with window_agg
all_pairs = thresholds.merge(window_agg, on=['cmdb_id', 'kpi_name'], how='left')

# fill missing numeric aggregates with zeros where appropriate
all_pairs['total_points_in_window'] = all_pairs['total_points_in_window'].fillna(0).astype(int)
all_pairs['anomaly_count_high'] = all_pairs['anomaly_count_high'].fillna(0).astype(int)
all_pairs['anomaly_count_low'] = all_pairs['anomaly_count_low'].fillna(0).astype(int)

# earliest timestamp: format to string or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    # x is tz-aware Timestamp
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

all_pairs['earliest_anomaly_timestamp_in_window'] = all_pairs['earliest_anomaly_timestamp_in_window'].apply(fmt_ts)

# keep required columns and compute total anomalies
all_pairs['total_anomaly_count'] = all_pairs['anomaly_count_high'] + all_pairs['anomaly_count_low']

result_cols = [
    'cmdb_id',
    'kpi_name',
    'total_points_in_window',
    'anomaly_count_high',
    'anomaly_count_low',
    'earliest_anomaly_timestamp_in_window',
    'P95',
    'P5',
    'max_value_in_window',
    'min_value_in_window',
    'total_anomaly_count'
]

result_df = all_pairs[result_cols].sort_values(by='total_anomaly_count', ascending=False).head(20).reset_index(drop=True)

# final display: drop the helper total_anomaly_count column from visible output if not desired,
# but keep it for sorting insight. We'll keep it for compactness.
top20_pairs = result_df

# total number of distinct (cmdb_id, kpi_name) pairs analyzed
total_pairs = int(thresholds.shape[0])

top20_pairs, total_pairs

```
Out[2]:
```
```python
# Plain-English summary based on the previous results in the kernel (top20_pairs, total_pairs)
try:
    df = top20_pairs.copy()
    total = total_pairs
except NameError:
    df = None
    total = None

if df is None or total is None:
    summary_text = "Previous results not found in the kernel. Unable to summarize."
else:
    # Basic counts
    top_n = df.shape[0]
    # Check if top rows share identical window stats
    same_points = df['total_points_in_window'].nunique() == 1
    same_anom = df['total_anomaly_count'].nunique() == 1
    common_points = int(df['total_points_in_window'].iloc[0])
    common_total_anom = int(df['total_anomaly_count'].iloc[0])
    common_earliest = df['earliest_anomaly_timestamp_in_window'].dropna().iloc[0] if df['earliest_anomaly_timestamp_in_window'].notna().any() else None

    # Prepare short list of first 5 pairs
    sample_rows = []
    for _, r in df.head(5).iterrows():
        sample_rows.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | points={int(r['total_points_in_window'])} "
            f"| high={int(r['anomaly_count_high'])} low={int(r['anomaly_count_low'])} "
            f"| P95={r['P95']} P5={r['P5']} | earliest={r['earliest_anomaly_timestamp_in_window']}"
        )

    summary_lines = [
        f"Metric scan summary:",
        f"- Total distinct (cmdb_id, kpi_name) pairs analyzed: {total}.",
        f"- Top {top_n} pairs (sorted by total anomalies):",
        f"  * All top {top_n} rows have {common_points} points within the 2022-03-21 22:00:00 to 22:30:00 (UTC+8) window."
        if same_points else f"  * Points-in-window vary across top {top_n} rows.",
        f"  * All top {top_n} rows have total_anomaly_count = {common_total_anom} (sum of high and low) "
        f"and earliest anomaly at {common_earliest}."
        if same_anom else "  * Total anomaly counts vary across the top rows.",
        "",
        "Sample of top pairs (first 5):"
    ]
    summary_lines += [f"  - {s}" for s in sample_rows]
    summary_lines += [
        "",
        "Interpretation notes:",
        "- P95 and P5 were computed globally per (cmdb_id, kpi_name) using the full metric series before window filtering.",
        "- Anomalies inside the window were flagged when value >= global P95 or value <= global P5.",
        "- The top pairs indicate many metrics were constant (P95 == P5 == observed value), causing both high and low anomaly counts to be triggered for every point."
    ]

    summary_text = "\n".join(summary_lines)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  total_points_in_window  anomaly_count_high  anomaly_count_low earliest_anomaly_timestamp_in_window      P95       P5  max_value_in_window  min_value_in_window  total_anomaly_count
0   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62
1   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62
2   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62
3            adservice                                        app.grpc.rr                      31                  31                 31             2022-03-21 22:00:00+0800    100.0    100.0                100.0                100.0                   62
4            adservice                                        app.grpc.sr                      31                  31                 31             2022-03-21 22:00:00+0800    100.0    100.0                100.0                100.0                   62
5            adservice                                     app.http.count                      31                  31                 31             2022-03-21 22:00:00+0800     24.0     24.0                 24.0                 24.0                   62
6   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62
7            adservice                                        app.http.rr                      31                  31                 31             2022-03-21 22:00:00+0800    100.0    100.0                100.0                100.0                   62
8            adservice                                        app.http.sr                      31                  31                 31             2022-03-21 22:00:00+0800    100.0    100.0                100.0                100.0                   62
9            adservice    runtime.java_lang_ClassLoading_LoadedClassCount                      31                  31                 31             2022-03-21 22:00:00+0800   5162.0   5162.0               5162.0               5162.0                   62
10           adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...                      31                  31                 31             2022-03-21 22:00:00+0800   5185.0   5185.0               5185.0               5185.0                   62
11           adservice  runtime.java_lang_ClassLoading_UnloadedClassCount                      31                  31                 31             2022-03-21 22:00:00+0800     23.0     23.0                 23.0                 23.0                   62
12           adservice             runtime.java_lang_ClassLoading_Verbose                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62
13           adservice  runtime.java_lang_Compilation_CompilationTimeM...                      31                  31                 31             2022-03-21 22:00:00+0800      1.0      1.0                  1.0                  1.0                   62
14           adservice  runtime.java_lang_Compilation_TotalCompilation...                      31                  31                 31             2022-03-21 22:00:00+0800  47032.0  47032.0              47032.0              47032.0                   62
15  shippingservice2-0          container.node-6.container_memory_failcnt                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62
16  shippingservice2-0  container.node-6.container_memory_failures.con...                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62
17  shippingservice2-0  container.node-6.container_memory_failures.con...                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62
18  shippingservice2-0  container.node-6.container_memory_failures.hie...                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62
19  shippingservice2-0  container.node-6.container_memory_failures.hie...                      31                  31                 31             2022-03-21 22:00:00+0800      0.0      0.0                  0.0                  0.0                   62, 5374)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse existing metric_df and tz if present, otherwise read and set tz
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timestamp tz-aware column
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using entire series (before window filter)
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(P95=lambda x: float(x.quantile(0.95)), P5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) Filter to incident window (inclusive)
start = pd.Timestamp('2022-03-21 22:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
window_df = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] <= end)].copy()

# 3) Merge thresholds into window rows and compute anomaly flags using global thresholds
window_df = window_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
window_df['anom_high'] = (window_df['value'] >= window_df['P95']).astype(int)
window_df['anom_low'] = (window_df['value'] <= window_df['P5']).astype(int)
window_df['anom_ts'] = window_df['ts'].where((window_df['anom_high'] == 1) | (window_df['anom_low'] == 1))

# Aggregate per (cmdb_id, kpi_name) within window
pair_agg = (
    window_df
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count_high=('anom_high', 'sum'),
        anomaly_count_low=('anom_low', 'sum'),
        earliest_anomaly_timestamp_in_window=('anom_ts', 'min'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
    )
    .reset_index()
)

# 4) Mark non_constant pairs: max != min and total_points_in_window > 0
pair_agg['non_constant'] = (pair_agg['total_points_in_window'] > 0) & (pair_agg['max_value_in_window'] != pair_agg['min_value_in_window'])

# Format earliest anomaly timestamps to string in UTC+8 or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

pair_agg['earliest_anomaly_timestamp_in_window'] = pair_agg['earliest_anomaly_timestamp_in_window'].apply(fmt_ts)

# Ensure numeric columns are ints where appropriate
pair_agg['anomaly_count_high'] = pair_agg['anomaly_count_high'].astype(int)
pair_agg['anomaly_count_low'] = pair_agg['anomaly_count_low'].astype(int)
pair_agg['total_points_in_window'] = pair_agg['total_points_in_window'].astype(int)

# 5) Aggregate per cmdb_id over only non_constant pairs
# First compute total distinct kpi count per cmdb_id from the whole file
total_kpi_per_cmdb = thresholds.groupby('cmdb_id', observed=True)['kpi_name'].nunique().rename('total_distinct_kpi_count').reset_index()

# Filter only non_constant pairs
nonconst_pairs = pair_agg[pair_agg['non_constant']].copy()
nonconst_pairs['pair_anomaly_sum'] = nonconst_pairs['anomaly_count_high'] + nonconst_pairs['anomaly_count_low']

# Aggregate per cmdb_id
agg_cmdb = (
    nonconst_pairs
    .groupby('cmdb_id', observed=True)
    .agg(
        non_constant_kpi_count=('kpi_name', 'nunique'),
        total_anomaly_count_non_constant=('pair_anomaly_sum', 'sum'),
        earliest_anomaly_timestamp_non_constant=('earliest_anomaly_timestamp_in_window', lambda x: min([t for t in x if t is not None]) if len([t for t in x if t is not None])>0 else None)
    )
    .reset_index()
)

# Merge total distinct kpi count (from whole series)
agg_cmdb = agg_cmdb.merge(total_kpi_per_cmdb, on='cmdb_id', how='left')

# 6) For each cmdb_id produce sample_kpis up to 10 items for non_constant pairs that had anomalies (pair_anomaly_sum>0),
# sorted by pair_anomaly_sum desc. Each item: (kpi_name, total_points_in_window, anomaly_count_high, anomaly_count_low)
sample_src = nonconst_pairs[nonconst_pairs['pair_anomaly_sum'] > 0].copy()
sample_src = sample_src.sort_values(['cmdb_id', 'pair_anomaly_sum'], ascending=[True, False])

sample_lists = (
    sample_src
    .groupby('cmdb_id', observed=True)
    .apply(lambda df: df.head(10)[['kpi_name','total_points_in_window','anomaly_count_high','anomaly_count_low']].apply(tuple, axis=1).tolist())
    .rename('sample_kpis')
    .reset_index()
)

# Merge sample lists into agg_cmdb
agg_cmdb = agg_cmdb.merge(sample_lists, on='cmdb_id', how='left')

# Fill NaN sample_kpis with empty list
agg_cmdb['sample_kpis'] = agg_cmdb['sample_kpis'].apply(lambda x: x if isinstance(x, list) else [])

# 7) Return top 20 cmdb_id rows sorted by total_anomaly_count_non_constant desc and total number of distinct cmdb_id analyzed
agg_cmdb = agg_cmdb.sort_values('total_anomaly_count_non_constant', ascending=False).reset_index(drop=True)
top20_cmdbs = agg_cmdb.head(20)[['cmdb_id','non_constant_kpi_count','total_anomaly_count_non_constant','earliest_anomaly_timestamp_non_constant','total_distinct_kpi_count','sample_kpis']]

total_cmdb_analyzed = int(thresholds['cmdb_id'].nunique())

# Display compact results
top20_cmdbs, total_cmdb_analyzed

```
Out[3]:
```
Summary (plain English):

- Total distinct cmdb_id analyzed: 59.

Top 20 cmdb_id by anomalies across non-constant KPIs (only key fields shown):
1. checkoutservice-0 — non_constant_kpi_count: 63; total_anomaly_count_non_constant: 650; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 144.
2. node-5 — non_constant_kpi_count: 41; total_anomaly_count_non_constant: 643; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 58.
3. checkoutservice-1 — non_constant_kpi_count: 63; total_anomaly_count_non_constant: 605; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 148.
4. checkoutservice-2 — non_constant_kpi_count: 59; total_anomaly_count_non_constant: 546; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 144.
5. frontend2-0 — non_constant_kpi_count: 73; total_anomaly_count_non_constant: 513; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 208.
6. adservice2 — non_constant_kpi_count: 51; total_anomaly_count_non_constant: 506; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 333.
7. checkoutservice2-0 — non_constant_kpi_count: 61; total_anomaly_count_non_constant: 498; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 152.
8. cartservice2-0 — non_constant_kpi_count: 40; total_anomaly_count_non_constant: 473; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 100.
9. node-4 — non_constant_kpi_count: 36; total_anomaly_count_non_constant: 442; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 59.
10. frontend-2 — non_constant_kpi_count: 65; total_anomaly_count_non_constant: 420; earliest anomaly: 2022-03-21 22:00:00+0800; total_distinct_kpi_count: 192.
(remaining top-20 entries include productcatalogservice-1, productcatalogservice-0, node-3, node-6, node-1, productcatalogservice2-0, currencyservice-0, emailservice-2, cartservice-2, emailservice-1 — see full table for exact counts.)

- For each cmdb_id the result includes up to 10 sample non-constant KPIs that contributed anomalies; each sample item is (kpi_name, total_points_in_window, anomaly_count_high, anomaly_count_low). Samples are limited to the most anomalous KPIs per cmdb_id.

Interpretation notes:
- Aggregation considered only non-constant KPIs (max != min in the 22:00–22:30 window and having points).
- Anomalies were counted based on global P95/P5 thresholds computed from the entire time series for each (cmdb_id, kpi_name) before window filtering.
- Earliest anomalies for top items all occurred at 2022-03-21 22:00:00+0800.
- Checkoutservice instances and several nodes show the highest anomaly counts among non-constant KPIs, indicating they had many KPIs crossing their global thresholds during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  non_constant_kpi_count  total_anomaly_count_non_constant earliest_anomaly_timestamp_non_constant  total_distinct_kpi_count                                        sample_kpis
0          checkoutservice-0                      63                               650                2022-03-21 22:00:00+0800                       144  [(mesh.source.checkoutservice.jaeger-collector...
1                     node-5                      41                               643                2022-03-21 22:00:00+0800                        58  [(system.udp.connect.num, 31, 31, 30), (system...
2          checkoutservice-1                      63                               605                2022-03-21 22:00:00+0800                       148  [(container.node-6.container_memory_working_se...
3          checkoutservice-2                      59                               546                2022-03-21 22:00:00+0800                       144  [(container.node-5.container_cpu_cfs_throttled...
4                frontend2-0                      73                               513                2022-03-21 22:00:00+0800                       208  [(container.node-6.container_memory_failures.c...
5                 adservice2                      51                               506                2022-03-21 22:00:00+0800                       333  [(runtime.jvm_threads_state.RUNNABLE, 31, 31, ...
6         checkoutservice2-0                      61                               498                2022-03-21 22:00:00+0800                       152  [(container.node-6.container_memory_cache, 31,...
7             cartservice2-0                      40                               473                2022-03-21 22:00:00+0800                       100  [(container.node-5.container_memory_failures.c...
8                     node-4                      36                               442                2022-03-21 22:00:00+0800                        59  [(system.process.zombie.num, 31, 31, 29), (sys...
9                 frontend-2                      65                               420                2022-03-21 22:00:00+0800                       192  [(container.node-5.container_cpu_cfs_throttled...
10   productcatalogservice-1                      46                               389                2022-03-21 22:00:00+0800                       120  [(container.node-6.container_memory_failures.c...
11   productcatalogservice-0                      46                               369                2022-03-21 22:00:00+0800                       120  [(container.node-6.container_memory_failures.c...
12                    node-3                      39                               360                2022-03-21 22:00:00+0800                        58  [(system.disk.pct_usage, 31, 30, 1), (system.i...
13                    node-6                      41                               331                2022-03-21 22:00:00+0800                        59  [(system.disk.total, 31, 5, 22), (system.fs.in...
14                    node-1                      40                               330                2022-03-21 22:00:00+0800                        59  [(system.process.zombie.num, 31, 31, 30), (sys...
15  productcatalogservice2-0                      38                               323                2022-03-21 22:00:00+0800                       104  [(container.node-6.container_file_descriptors,...
16         currencyservice-0                      30                               315                2022-03-21 22:00:00+0800                        80  [(container.node-6.container_cpu_cfs_throttled...
17            emailservice-2                      25                               314                2022-03-21 22:00:00+0800                        78  [(mesh.source.emailservice.jaeger-collector.is...
18             cartservice-2                      33                               295                2022-03-21 22:00:00+0800                       100  [(container.node-6.container_cpu_cfs_throttled...
19            emailservice-1                      25                               282                2022-03-21 22:00:00+0800                        74  [(mesh.source.emailservice.jaeger-collector.is..., 59)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse tz if available
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# read trace data
trace_df = pd.read_csv("trace.csv")

# create timezone-aware timestamp column in Asia/Shanghai (UTC+8)
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) global P95 and P5 per (cmdb_id, trace_name) using full series
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .agg(P95=lambda x: float(x.quantile(0.95)), P5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) filter to incident window (inclusive)
start = pd.Timestamp('2022-03-21 22:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
trace_window = trace_df[(trace_df['ts'] >= start) & (trace_df['ts'] <= end)].copy()

# 3) merge thresholds into window rows and compute anomaly flags (use global thresholds)
trace_window = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
trace_window['anom_high'] = (trace_window['value'] >= trace_window['P95']).astype(int)
trace_window['anom_low'] = (trace_window['value'] <= trace_window['P5']).astype(int)
trace_window['anom_ts'] = trace_window['ts'].where((trace_window['anom_high'] == 1) | (trace_window['anom_low'] == 1))

# aggregate per pair within window
pair_agg = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'], observed=True)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count_high=('anom_high', 'sum'),
        anomaly_count_low=('anom_low', 'sum'),
        earliest_anomaly_timestamp_in_window=('anom_ts', 'min'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
    )
    .reset_index()
)

# 2b) include pairs that had no points in window (left join with full thresholds)
all_pairs = trace_thresholds.merge(pair_agg, on=['cmdb_id', 'trace_name'], how='left')

# fill missing numeric aggregates with zeros / NAs handled
all_pairs['total_points_in_window'] = all_pairs['total_points_in_window'].fillna(0).astype(int)
all_pairs['anomaly_count_high'] = all_pairs['anomaly_count_high'].fillna(0).astype(int)
all_pairs['anomaly_count_low'] = all_pairs['anomaly_count_low'].fillna(0).astype(int)

# format earliest anomaly timestamp
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

all_pairs['earliest_anomaly_timestamp_in_window'] = all_pairs['earliest_anomaly_timestamp_in_window'].apply(fmt_ts)

# ensure max/min columns exist (may be NaN for zero-point pairs)
all_pairs['max_value_in_window'] = all_pairs['max_value_in_window'].where(~all_pairs['max_value_in_window'].isna(), None)
all_pairs['min_value_in_window'] = all_pairs['min_value_in_window'].where(~all_pairs['min_value_in_window'].isna(), None)

# compute total anomaly count for sorting
all_pairs['total_anomaly_count'] = all_pairs['anomaly_count_high'] + all_pairs['anomaly_count_low']

# 4) return top 20 pairs sorted by total_anomaly_count desc with requested columns
result_cols = [
    'cmdb_id',
    'trace_name',
    'total_points_in_window',
    'anomaly_count_high',
    'anomaly_count_low',
    'earliest_anomaly_timestamp_in_window',
    'P95',
    'P5',
    'max_value_in_window',
    'min_value_in_window',
    'total_anomaly_count'
]

top20 = all_pairs.sort_values('total_anomaly_count', ascending=False)[result_cols].head(20).reset_index(drop=True)

# 5) total number of distinct (cmdb_id, trace_name) pairs analyzed
total_pairs = int(trace_thresholds.shape[0])

# display compact results: top20 dataframe and the total count
top20, total_pairs

```
Out[4]:
```
```python
summary_text = (
    "Trace anomaly scan summary:\n\n"
    "- Total distinct (cmdb_id, trace_name) pairs analyzed: 1,056.\n\n"
    "- Top 20 pairs by anomaly count (brief highlights):\n"
    "  * cartservice-0 | trace.self.error_rate — 30 points in window, high=30, low=30, earliest=2022-03-21 22:00:00+0800, P95=0.000, P5=0.000, max=0.000, min=0.000\n"
    "  * adservice-0 | trace.from_frontend-2.error_rate — 30 pts, high=30, low=30, earliest=2022-03-21 22:00:00+0800, P95=0.000, P5=0.000\n"
    "  * productcatalogservice-1 | several trace.from_*.error_rate entries — each 30 pts, high=30, low=30, earliest=2022-03-21 22:00:00+0800, P95=0.000, P5=0.000\n"
    "  * checkoutservice2-0 | multiple trace.to_*.error_rate entries — same pattern (30 pts, high=30, low=30)\n"
    "  * cartservice2-0 | trace.from_frontend2-0.duration_p95 — 30 pts, high=30, low=30, P95=0.001, P5=0.001, max/min=0.001\n\n"
    "- Observations and interpretation:\n"
    "  * Most top entries are error_rate metrics that are constant zeros across the window (P95 == P5 == 0.000). Because thresholds were computed globally, every value equals both the P95 and P5, causing both high and low anomaly counts to increment for each point (e.g., high=30 and low=30 → total=60).\n"
    "  * This artifact (constant series equal to threshold) dominates the ranking and explains why many pairs show identical counts and earliest anomaly at 2022-03-21 22:00:00+0800.\n"
    "  * A few non-zero constant traces (e.g., duration_p95 = 0.001) show the same behavior when P95==P5==observed value.\n\n"
    "Recommendation: review constant-valued trace metrics (especially error_rate) before treating these as true anomalies—consider ignoring constant series or using strict comparison (> P95 / < P5) to avoid flagging equality as both high and low anomalies."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                     trace_name  total_points_in_window  anomaly_count_high  anomaly_count_low earliest_anomaly_timestamp_in_window    P95     P5  max_value_in_window  min_value_in_window  total_anomaly_count
0             cartservice-0                          trace.self.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
1               adservice-0               trace.from_frontend-2.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
2   productcatalogservice-1  trace.from_recommendationservice-1.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
3   productcatalogservice-1  trace.from_recommendationservice-2.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
4   productcatalogservice-1               trace.from_frontend-2.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
5   productcatalogservice-0               trace.from_frontend-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
6   productcatalogservice-1               trace.from_frontend-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
7   productcatalogservice-1  trace.from_recommendationservice-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
8         paymentservice2-0       trace.from_checkoutservice2-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
9        checkoutservice2-0             trace.to_cartservice2-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
10       checkoutservice2-0          trace.to_paymentservice2-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
11       checkoutservice2-0   trace.to_productcatalogservice2-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
12       checkoutservice2-0         trace.to_currencyservice2-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
13       checkoutservice2-0            trace.to_emailservice2-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
14       checkoutservice2-0         trace.to_shippingservice2-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
15            cartservice-1               trace.from_frontend-2.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
16            cartservice-1                          trace.self.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
17           cartservice2-0       trace.from_checkoutservice2-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60
18           cartservice2-0            trace.from_frontend2-0.duration_p95                      30                  30                 30             2022-03-21 22:00:00+0800  0.001  0.001                0.001                0.001                   60
19           cartservice2-0              trace.from_frontend2-0.error_rate                      30                  30                 30             2022-03-21 22:00:00+0800  0.000  0.000                0.000                0.000                   60, 1056)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse variables if present
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware timestamp column in Asia/Shanghai (UTC+8)
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) global P95 and P5 per (cmdb_id, trace_name) using full series (must be computed BEFORE window filtering)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .agg(P95=lambda x: float(x.quantile(0.95)), P5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) filter to incident window (inclusive)
start = pd.Timestamp('2022-03-21 22:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
trace_window = trace_df[(trace_df['ts'] >= start) & (trace_df['ts'] <= end)].copy()

# 3) merge thresholds into window rows and compute strict-anomaly flags (>) and (<)
trace_window = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
trace_window['anom_high'] = (trace_window['value'] > trace_window['P95']).astype(int)
trace_window['anom_low'] = (trace_window['value'] < trace_window['P5']).astype(int)
trace_window['anom_ts'] = trace_window['ts'].where((trace_window['anom_high'] == 1) | (trace_window['anom_low'] == 1))

# aggregate per (cmdb_id, trace_name) within window
pair_agg = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'], observed=True)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count_high=('anom_high', 'sum'),
        anomaly_count_low=('anom_low', 'sum'),
        earliest_anomaly_ts=('anom_ts', 'min'),  # keep as Timestamp for min across pairs
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
    )
    .reset_index()
)

# 4) Mark non_constant if max != min and total_points_in_window > 0
pair_agg['non_constant'] = (pair_agg['total_points_in_window'] > 0) & (pair_agg['max_value_in_window'] != pair_agg['min_value_in_window'])

# compute pair anomaly sum
pair_agg['pair_anomaly_sum'] = pair_agg['anomaly_count_high'] + pair_agg['anomaly_count_low']

# 5) Aggregate per cmdb_id over only non_constant pairs
nonconst = pair_agg[pair_agg['non_constant']].copy()

# non_constant_trace_count per cmdb_id
agg = (
    nonconst
    .groupby('cmdb_id', observed=True)
    .agg(
        non_constant_trace_count=('trace_name', 'nunique'),
        total_anomaly_count_non_constant=('pair_anomaly_sum', 'sum'),
        earliest_anomaly_ts_non_constant=('earliest_anomaly_ts', lambda x: x.min() if x.notna().any() else pd.NaT)
    )
    .reset_index()
)

# 6) sample list up to 10 of most anomalous non_constant traces per cmdb_id
sample_src = nonconst.copy()
# sort by pair_anomaly_sum desc within each cmdb_id
sample_src = sample_src.sort_values(['cmdb_id', 'pair_anomaly_sum'], ascending=[True, False])

sample_lists = (
    sample_src[sample_src['pair_anomaly_sum'] > 0]
    .groupby('cmdb_id', observed=True)
    .apply(lambda df: df.head(10)[['trace_name','total_points_in_window','anomaly_count_high','anomaly_count_low']].apply(tuple, axis=1).tolist())
    .rename('sample_traces')
    .reset_index()
)

agg = agg.merge(sample_lists, on='cmdb_id', how='left')
agg['sample_traces'] = agg['sample_traces'].apply(lambda x: x if isinstance(x, list) else [])

# format earliest_anomaly_ts_non_constant to string with timezone or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

agg['earliest_anomaly_timestamp_non_constant'] = agg['earliest_anomaly_ts_non_constant'].apply(fmt_ts)
agg = agg.drop(columns=['earliest_anomaly_ts_non_constant'])

# 7) top 20 cmdb_id rows sorted by total_anomaly_count_non_constant desc; include requested columns
top20_cmdb = agg.sort_values('total_anomaly_count_non_constant', ascending=False).reset_index(drop=True).head(20)
top20_cmdb = top20_cmdb[['cmdb_id','non_constant_trace_count','total_anomaly_count_non_constant','earliest_anomaly_timestamp_non_constant','sample_traces']]

# total number of distinct (cmdb_id, trace_name) pairs analyzed
total_pairs = int(trace_thresholds.shape[0])

# display compact results
top20_cmdb, total_pairs

```
Out[5]:
```
Summary (plain English):

- Total distinct (cmdb_id, trace_name) pairs analyzed: 1,056.

Top 20 cmdb_id by anomalies across non-constant traces (fields shown: non_constant_trace_count, total_anomaly_count_non_constant, earliest anomaly):
1. frontend-2 — 66 non-constant traces; total anomalies 187; earliest anomaly 2022-03-21 22:00:00+0800  
2. frontend-0 — 66 non-constant traces; total anomalies 183; earliest 2022-03-21 22:00:00+0800  
3. frontend2-0 — 26 non-constant traces; total anomalies 87; earliest 2022-03-21 22:00:00+0800  
4. checkoutservice2-0 — 27 non-constant traces; total anomalies 83; earliest 2022-03-21 22:02:00+0800  
5. checkoutservice-1 — 49 non-constant traces; total anomalies 81; earliest 2022-03-21 22:00:00+0800  
6. checkoutservice-2 — 44 non-constant traces; total anomalies 76; earliest 2022-03-21 22:00:00+0800  
7. checkoutservice-0 — 49 non-constant traces; total anomalies 76; earliest 2022-03-21 22:02:00+0800  
8. productcatalogservice-2 — 24 non-constant traces; total anomalies 57; earliest 2022-03-21 22:00:00+0800  
9. productcatalogservice-0 — 24 non-constant traces; total anomalies 54; earliest 2022-03-21 22:00:00+0800  
10. productcatalogservice-1 — 24 non-constant traces; total anomalies 54; earliest 2022-03-21 22:00:00+0800  
11. recommendationservice-0 — 18 non-constant traces; total anomalies 46; earliest 2022-03-21 22:00:00+0800  
12. recommendationservice-2 — 18 non-constant traces; total anomalies 46; earliest 2022-03-21 22:00:00+0800  
13. recommendationservice2-0 — 15 non-constant traces; total anomalies 40; earliest 2022-03-21 22:00:00+0800  
14. recommendationservice-1 — 18 non-constant traces; total anomalies 38; earliest 2022-03-21 22:00:00+0800  
15. currencyservice-0 — 14 non-constant traces; total anomalies 34; earliest 2022-03-21 22:00:00+0800  
16. currencyservice-1 — 13 non-constant traces; total anomalies 34; earliest 2022-03-21 22:00:00+0800  
17. currencyservice-2 — 14 non-constant traces; total anomalies 32; earliest 2022-03-21 22:00:00+0800  
18. shippingservice-1 — 13 non-constant traces; total anomalies 26; earliest 2022-03-21 22:01:00+0800  
19. shippingservice-0 — 14 non-constant traces; total anomalies 26; earliest 2022-03-21 22:00:00+0800  
20. shippingservice-2 — 12 non-constant traces; total anomalies 23; earliest 2022-03-21 22:00:00+0800

- For each cmdb_id the output also contains up to 10 sample non-constant traces (format: (trace_name, total_points_in_window, anomaly_count_high, anomaly_count_low)), prioritized by highest anomaly sums.

Method and important notes:
- Global P95 and P5 were computed per (cmdb_id, trace_name) using the full series (before window filtering).
- The incident window used: 2022-03-21 22:00:00 to 2022-03-21 22:30:00 (UTC+8).
- Strict inequalities were used for anomaly detection (value > P95 and value < P5) to avoid equality artifacts from constant series.
- Most top cmdb_id entries are frontends, checkout services, productcatalog and recommendation services — these show the largest counts of non-constant trace anomalies in the window.
- Many earliest anomalies occur at or immediately after 22:00, suggesting the issue was present at the start of the analyzed window.

Suggested next steps:
- Inspect the sample trace names for the top cmdb_id rows (e.g., trace.from_frontend.* and trace.self.*) to identify which trace paths or spans degraded.
- Correlate these trace anomalies with metric and log anomalies for the same components to locate likely root causes.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  non_constant_trace_count  total_anomaly_count_non_constant earliest_anomaly_timestamp_non_constant                                      sample_traces
0                 frontend-2                        66                               187                2022-03-21 22:00:00+0800  [(trace.from_root.duration_mean, 30, 2, 2), (t...
1                 frontend-0                        66                               183                2022-03-21 22:00:00+0800  [(trace.from_root.duration_mean, 30, 2, 2), (t...
2                frontend2-0                        26                                87                2022-03-21 22:00:00+0800  [(trace.from_root.duration_mean, 30, 2, 2), (t...
3         checkoutservice2-0                        27                                83                2022-03-21 22:02:00+0800  [(trace.from_frontend2-0.duration_mean, 30, 2,...
4          checkoutservice-1                        49                                81                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_mean, 10, 1, ...
5          checkoutservice-2                        44                                76                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_mean, 10, 1, ...
6          checkoutservice-0                        49                                76                2022-03-21 22:02:00+0800  [(trace.from_frontend-0.duration_mean, 10, 1, ...
7    productcatalogservice-2                        24                                57                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_mean, 30, 2, ...
8    productcatalogservice-0                        24                                54                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_mean, 30, 2, ...
9    productcatalogservice-1                        24                                54                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_mean, 30, 2, ...
10   recommendationservice-0                        18                                46                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_mean, 30, 2, ...
11   recommendationservice-2                        18                                46                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_mean, 30, 2, ...
12  recommendationservice2-0                        15                                40                2022-03-21 22:00:00+0800  [(trace.from_frontend2-0.duration_mean, 30, 2,...
13   recommendationservice-1                        18                                38                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_mean, 30, 2, ...
14         currencyservice-0                        14                                34                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_mean, 30, 2, ...
15         currencyservice-1                        13                                34                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.row_count, 30, 2, 2), ...
16         currencyservice-2                        14                                32                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.row_count, 30, 2, 2), ...
17         shippingservice-1                        13                                26                2022-03-21 22:01:00+0800  [(trace.from_frontend-2.duration_mean, 28, 2, ...
18         shippingservice-0                        14                                26                2022-03-21 22:00:00+0800  [(trace.from_frontend-2.duration_mean, 26, 2, ...
19         shippingservice-2                        12                                23                2022-03-21 22:00:00+0800  [(trace.from_frontend-0.duration_p95, 27, 2, 2..., 1056)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse tz if present
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Read logs and error_logs
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# create timezone-aware timestamp column in Asia/Shanghai (UTC+8)
if 'ts' not in log_df.columns:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) global P95 and P5 per (cmdb_id, log_name) using full series (before window filter)
log_thresholds = (
    log_df
    .groupby(['cmdb_id', 'log_name'], observed=True)['value']
    .agg(P95=lambda x: float(x.quantile(0.95)), P5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) filter to incident window (inclusive)
start = pd.Timestamp('2022-03-21 22:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
log_window = log_df[(log_df['ts'] >= start) & (log_df['ts'] <= end)].copy()

# 3) merge thresholds and compute strict anomaly flags (>) and (<)
log_window = log_window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')
log_window['anom_high'] = (log_window['value'] > log_window['P95']).astype(int)
log_window['anom_low'] = (log_window['value'] < log_window['P5']).astype(int)
log_window['anom_ts'] = log_window['ts'].where((log_window['anom_high'] == 1) | (log_window['anom_low'] == 1))

# aggregate per (cmdb_id, log_name) within window
pair_agg = (
    log_window
    .groupby(['cmdb_id', 'log_name'], observed=True)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count_high=('anom_high', 'sum'),
        anomaly_count_low=('anom_low', 'sum'),
        earliest_anomaly_ts=('anom_ts', 'min'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
    )
    .reset_index()
)

# 4) mark non_constant: max != min and total_points_in_window > 0
pair_agg['non_constant'] = (pair_agg['total_points_in_window'] > 0) & (pair_agg['max_value_in_window'] != pair_agg['min_value_in_window'])
pair_agg['pair_anomaly_sum'] = pair_agg['anomaly_count_high'] + pair_agg['anomaly_count_low']

# 5) aggregate per cmdb_id over only non_constant pairs
nonconst = pair_agg[pair_agg['non_constant']].copy()

agg = (
    nonconst
    .groupby('cmdb_id', observed=True)
    .agg(
        non_constant_log_count=('log_name', 'nunique'),
        total_anomaly_count_non_constant=('pair_anomaly_sum', 'sum'),
        earliest_anom_ts_non_constant=('earliest_anomaly_ts', lambda x: x.min() if x.notna().any() else pd.NaT)
    )
    .reset_index()
)

# 6) sample list up to 10 of most anomalous non_constant logs per cmdb_id
sample_src = nonconst.copy()
sample_src = sample_src.sort_values(['cmdb_id', 'pair_anomaly_sum'], ascending=[True, False])

sample_lists = (
    sample_src[sample_src['pair_anomaly_sum'] > 0]
    .groupby('cmdb_id', observed=True)
    .apply(lambda df: df.head(10)[['log_name','total_points_in_window','anomaly_count_high','anomaly_count_low']].apply(tuple, axis=1).tolist())
    .rename('sample_logs')
    .reset_index()
)

agg = agg.merge(sample_lists, on='cmdb_id', how='left')
agg['sample_logs'] = agg['sample_logs'].apply(lambda x: x if isinstance(x, list) else [])

# format earliest anomaly timestamp to string in UTC+8 or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

agg['earliest_anomaly_timestamp_non_constant'] = agg['earliest_anom_ts_non_constant'].apply(fmt_ts)
agg = agg.drop(columns=['earliest_anom_ts_non_constant'])

# 7) top 20 cmdb_id rows sorted by total_anomaly_count_non_constant desc and total pairs analyzed
top20_cmdb = agg.sort_values('total_anomaly_count_non_constant', ascending=False).reset_index(drop=True).head(20)
top20_cmdb = top20_cmdb[['cmdb_id','non_constant_log_count','total_anomaly_count_non_constant','earliest_anomaly_timestamp_non_constant','sample_logs']]

total_pairs = int(log_thresholds.shape[0])
error_logs_has_rows = int(error_logs_df.shape[0]) > 0

# compact display
top20_cmdb, total_pairs, error_logs_has_rows

```
Out[6]:
```
```python
summary_text = (
    "Log anomaly scan summary:\n\n"
    "- Total distinct (cmdb_id, log_name) pairs analyzed: 62.\n"
    "- error_logs.csv contains no rows.\n\n"
    "- Top findings (top 20 cmdb_id by anomalies on non-constant logs):\n"
    "  1) cartservice-0 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 31 pts, high=2, low=2)\n"
    "  2) currencyservice-0 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:15:00+0800; sample_logs: (log.row_count, 31 pts, high=2, low=2)\n"
    "  3) currencyservice-1 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:14:00+0800; sample_logs: (log.row_count, 31 pts, high=2, low=2)\n"
    "  4) currencyservice-2 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:15:00+0800; sample_logs: (log.row_count, 31 pts, high=2, low=2)\n"
    "  5) frontend-2 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:05:00+0800; sample_logs: (log.row_count, 30 pts, high=2, low=2)\n"
    "  6) frontend-0 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 31 pts, high=2, low=2)\n"
    "  7) productcatalogservice-2 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 30 pts, high=2, low=2)\n"
    "  8) shippingservice-2 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:05:00+0800; sample_logs: (log.row_count, 31 pts, high=2, low=2)\n"
    "  9) shippingservice-1 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:06:00+0800; sample_logs: (log.row_count, 30 pts, high=2, low=2)\n"
    " 10) shippingservice-0 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 30 pts, high=2, low=2)\n"
    " 11) productcatalogservice-1 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 30 pts, high=2, low=2)\n"
    " 12) productcatalogservice-0 — non_constant_log_count: 1; total_anomaly_count_non_constant: 4; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 30 pts, high=2, low=2)\n"
    " 13) cartservice-2 — non_constant_log_count: 1; total_anomaly_count_non_constant: 3; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 31 pts, high=2, low=1)\n"
    " 14) cartservice-1 — non_constant_log_count: 1; total_anomaly_count_non_constant: 3; earliest: 2022-03-21 22:02:00+0800; sample_logs: (log.row_count, 30 pts, high=2, low=1)\n"
    " 15) recommendationservice-2 — non_constant_log_count: 1; total_anomaly_count_non_constant: 3; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 30 pts, high=2, low=1)\n"
    " 16) recommendationservice-1 — non_constant_log_count: 1; total_anomaly_count_non_constant: 3; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 31 pts, high=2, low=1)\n"
    " 17) recommendationservice-0 — non_constant_log_count: 1; total_anomaly_count_non_constant: 3; earliest: 2022-03-21 22:00:00+0800; sample_logs: (log.row_count, 31 pts, high=1, low=2)\n"
    " 18) paymentservice-1 — non_constant_log_count: 1; total_anomaly_count_non_constant: 2; earliest: 2022-03-21 22:03:00+0800; sample_logs: (log.row_count, 20 pts, high=1, low=1)\n"
    " 19) checkoutservice-0 — non_constant_log_count: 1; total_anomaly_count_non_constant: 1; earliest: 2022-03-21 22:03:00+0800; sample_logs: (log.row_count, 20 pts, high=1, low=0)\n"
    " 20) adservice-0 — non_constant_log_count: 1; total_anomaly_count_non_constant: 1; earliest: 2022-03-21 22:22:00+0800; sample_logs: (log.row_count, 30 pts, high=0, low=1)\n\n"
    "- Observations:\n"
    "  * All non-constant log anomalies are on log.row_count (the per-minute row counts) for various services.\n"
    "  * Many services show small but non-zero anomaly counts (mostly 3–4 anomalies over the 22:00–22:30 window).\n"
    "  * Earliest anomalies occur between 22:00 and ~22:15 (UTC+8) depending on service.\n\n"
    "- Recommendation:\n"
    "  * Inspect log.row_count changes for the listed services (sample_logs) to understand traffic or sampling changes around the incident window.\n"
    "  * error_logs.csv is empty, so no raw error messages were captured in that file for this time range."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  non_constant_log_count  total_anomaly_count_non_constant earliest_anomaly_timestamp_non_constant                  sample_logs
0             cartservice-0                       1                                 4                2022-03-21 22:00:00+0800  [(log.row_count, 31, 2, 2)]
1         currencyservice-0                       1                                 4                2022-03-21 22:15:00+0800  [(log.row_count, 31, 2, 2)]
2         currencyservice-1                       1                                 4                2022-03-21 22:14:00+0800  [(log.row_count, 31, 2, 2)]
3         currencyservice-2                       1                                 4                2022-03-21 22:15:00+0800  [(log.row_count, 31, 2, 2)]
4                frontend-2                       1                                 4                2022-03-21 22:05:00+0800  [(log.row_count, 30, 2, 2)]
5                frontend-0                       1                                 4                2022-03-21 22:00:00+0800  [(log.row_count, 31, 2, 2)]
6   productcatalogservice-2                       1                                 4                2022-03-21 22:00:00+0800  [(log.row_count, 30, 2, 2)]
7         shippingservice-2                       1                                 4                2022-03-21 22:05:00+0800  [(log.row_count, 31, 2, 2)]
8         shippingservice-1                       1                                 4                2022-03-21 22:06:00+0800  [(log.row_count, 30, 2, 2)]
9         shippingservice-0                       1                                 4                2022-03-21 22:00:00+0800  [(log.row_count, 30, 2, 2)]
10  productcatalogservice-1                       1                                 4                2022-03-21 22:00:00+0800  [(log.row_count, 30, 2, 2)]
11  productcatalogservice-0                       1                                 4                2022-03-21 22:00:00+0800  [(log.row_count, 30, 2, 2)]
12            cartservice-2                       1                                 3                2022-03-21 22:00:00+0800  [(log.row_count, 31, 2, 1)]
13            cartservice-1                       1                                 3                2022-03-21 22:02:00+0800  [(log.row_count, 30, 2, 1)]
14  recommendationservice-2                       1                                 3                2022-03-21 22:00:00+0800  [(log.row_count, 30, 2, 1)]
15  recommendationservice-1                       1                                 3                2022-03-21 22:00:00+0800  [(log.row_count, 31, 2, 1)]
16  recommendationservice-0                       1                                 3                2022-03-21 22:00:00+0800  [(log.row_count, 31, 1, 2)]
17         paymentservice-1                       1                                 2                2022-03-21 22:03:00+0800  [(log.row_count, 20, 1, 1)]
18        checkoutservice-0                       1                                 1                2022-03-21 22:03:00+0800  [(log.row_count, 20, 1, 0)]
19              adservice-0                       1                                 1                2022-03-21 22:22:00+0800  [(log.row_count, 30, 0, 1)], 62, False)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# reuse existing metric_df and tz if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware timestamp column in Asia/Shanghai (UTC+8)
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Node candidates
nodes = ['node-5','node-4','node-3','node-6','node-1']

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) for selected nodes (using full series)
node_thresholds = (
    metric_df[metric_df['cmdb_id'].isin(nodes)]
    .groupby(['cmdb_id','kpi_name'], observed=True)['value']
    .agg(P95=lambda x: float(x.quantile(0.95)), P5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# total number of (cmdb_id, kpi_name) pairs analyzed for these nodes (before filtering constants)
total_pairs_for_nodes = int(node_thresholds.shape[0])

# 2) Filter rows to incident window (inclusive)
start = pd.Timestamp('2022-03-21 22:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
window_df = metric_df[
    (metric_df['ts'] >= start) & (metric_df['ts'] <= end) & (metric_df['cmdb_id'].isin(nodes))
].copy()

# 3) Merge thresholds into window rows so we can compute breaches using global thresholds
window_df = window_df.merge(node_thresholds, on=['cmdb_id','kpi_name'], how='left')

# compute per-row breach flags (strict > and < as requested)
window_df['above_P95'] = window_df['value'] > window_df['P95']
window_df['below_P5'] = window_df['value'] < window_df['P5']

# aggregate per pair within window: total points, max/min
pair_window_agg = (
    window_df
    .groupby(['cmdb_id','kpi_name'], observed=True)
    .agg(
        total_points_in_window=('value','size'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
    )
    .reset_index()
)

# earliest timestamps for breaches
earliest_above = (
    window_df[window_df['above_P95']]
    .groupby(['cmdb_id','kpi_name'], observed=True)['ts']
    .min()
    .reset_index()
    .rename(columns={'ts':'earliest_timestamp_above_P95'})
)

earliest_below = (
    window_df[window_df['below_P5']]
    .groupby(['cmdb_id','kpi_name'], observed=True)['ts']
    .min()
    .reset_index()
    .rename(columns={'ts':'earliest_timestamp_below_P5'})
)

# combine aggregates with thresholds (include pairs that may have no points in window)
pairs = node_thresholds.merge(pair_window_agg, on=['cmdb_id','kpi_name'], how='left')

# fill NaNs for pairs with no points
pairs['total_points_in_window'] = pairs['total_points_in_window'].fillna(0).astype(int)
pairs['max_value_in_window'] = pairs['max_value_in_window'].where(~pairs['max_value_in_window'].isna(), None)
pairs['min_value_in_window'] = pairs['min_value_in_window'].where(~pairs['min_value_in_window'].isna(), None)

# merge earliest breach timestamps
pairs = pairs.merge(earliest_above, on=['cmdb_id','kpi_name'], how='left')
pairs = pairs.merge(earliest_below, on=['cmdb_id','kpi_name'], how='left')

# format earliest timestamps to strings in UTC+8 or None
def fmt_ts(x):
    if pd.isna(x) or x is None:
        return None
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

pairs['earliest_timestamp_above_P95'] = pairs['earliest_timestamp_above_P95'].apply(fmt_ts)
pairs['earliest_timestamp_below_P5'] = pairs['earliest_timestamp_below_P5'].apply(fmt_ts)

# 4) Compute severity per rules using strict breaches (presence indicated by earliest timestamp not null)
def compute_severity(row):
    P95 = float(row['P95'])
    P5 = float(row['P5'])
    maxv = row['max_value_in_window']
    minv = row['min_value_in_window']
    # if no points in window, severity 0
    if row['total_points_in_window'] == 0 or maxv is None or minv is None:
        return 0.0
    sev_high = 0.0
    sev_low = 0.0
    # breach above if any value > P95 (earliest_timestamp_above_P95 not null)
    if row['earliest_timestamp_above_P95'] is not None:
        denom = max(abs(P95), 1e-9)
        sev_high = (float(maxv) - P95) / denom
    # breach below if any value < P5
    if row['earliest_timestamp_below_P5'] is not None:
        denom = max(abs(P5), 1e-9)
        sev_low = (P5 - float(minv)) / denom
    return float(max(sev_high, sev_low))

pairs['severity'] = pairs.apply(compute_severity, axis=1)

# 5) Exclude constant series (max == min) and also require total_points_in_window > 0
# Note: pairs with None for max/min have total_points_in_window==0 and will be excluded
filtered = pairs[
    (pairs['total_points_in_window'] > 0) &
    (pairs['max_value_in_window'].notna()) &
    (pairs['min_value_in_window'].notna()) &
    (pairs['max_value_in_window'] != pairs['min_value_in_window'])
].copy()

# 6) Prepare final columns and sort by severity desc, limit to top 20
final_cols = [
    'cmdb_id',
    'kpi_name',
    'total_points_in_window',
    'P95',
    'P5',
    'max_value_in_window',
    'min_value_in_window',
    'earliest_timestamp_above_P95',
    'earliest_timestamp_below_P5',
    'severity'
]

top20 = filtered.sort_values('severity', ascending=False).head(20).reset_index(drop=True)

# keep only requested columns
top20 = top20[final_cols]

# compact display: top20 dataframe and total number of pairs analyzed for these nodes
top20, total_pairs_for_nodes

```
Out[7]:
```
Summary of node-level severity (nodes analyzed: node-5, node-4, node-3, node-6, node-1)

- Total (cmdb_id, kpi_name) pairs computed for these nodes (before excluding constant series): 293.

- Top affected node: node-5 — highest severity scores driven by I/O metrics:
  - system.io.rkb_s spiked to 100 (earliest breach 2022-03-21 22:03:00+08).
  - system.io.r_s, system.io.r_await, system.io.avg_q_sz, system.io.util, system.io.await, system.io.w_await all show large breaches (earliest breaches mostly ~22:03–22:14+08).
  - system.cpu.iowait on node-5 rose (max ~4.10) with earliest breach ~22:13+08.
  - Interpretation: strong disk I/O pressure on node-5 during the incident window.

- node-3 also shows large I/O spikes:
  - system.io.rkb_s reached 246 (earliest breach 22:16+08).
  - system.io.r_s reached 18 (earliest breach 22:16+08).
  - Some elevated queue size and retrans metrics present but smaller than node-5.

- node-6 shows notable network and I/O signals:
  - system.io.rkb_s and system.io.r_s spiked (max 480 and 116.5 respectively, earliest ~22:04+08).
  - system.net.tcp.retrans_segs shows high retransmissions (earliest ~22:02+08), indicating network packet retransmission on node-6.

- node-1 shows:
  - system.process.zombie.num = 1 at ~22:17+08 (indicates process issues).
  - system.net.tcp.retrans_segs elevated (max ~5.37, earliest ~22:06+08).
  - system.cpu.iowait elevated with both above- and below-threshold events (earliest below-P5 at 22:13+08 and above-P95 at 22:30+08).

- node-4: smaller but present signal for system.process.zombie.num (1 at 22:04+08).

- Most breaches are above P95 (earliest_timestamp_above_P95 populated); earliest breach times cluster between ~22:02 and ~22:16 (UTC+8), depending on metric and node.

- Caveat on severity scaling: many P95 values are 0 (so relative severity becomes very large when max > 0). Use absolute maxima as well when judging impact (e.g., I/O KB/s, io.util percent, iowait seconds/percent, retransmission counts) — raw maxima listed above indicate real spikes in disk I/O and some TCP retransmissions.

- Overall conclusion: the strongest signals point to disk I/O pressure (node-5 primary, node-6 and node-3 also affected) accompanied by network retransmissions (node-6 and node-1) and some process-level anomalies (zombie processes on node-1 and node-4). These nodes are high-priority targets for investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                     kpi_name  total_points_in_window    P95    P5  max_value_in_window  min_value_in_window earliest_timestamp_above_P95 earliest_timestamp_below_P5      severity
0   node-5              system.io.rkb_s                      31   0.00  0.00               100.00                 0.00     2022-03-21 22:03:00+0800                        None  1.000000e+11
1   node-5                system.io.r_s                      31   0.00  0.00                 1.00                 0.00     2022-03-21 22:03:00+0800                        None  1.000000e+09
2   node-4    system.process.zombie.num                      31   0.00  0.00                 1.00                 0.00     2022-03-21 22:04:00+0800                        None  1.000000e+09
3   node-5            system.io.r_await                      31   0.00  0.00                 1.00                 0.00     2022-03-21 22:03:00+0800                        None  1.000000e+09
4   node-1    system.process.zombie.num                      31   0.00  0.00                 1.00                 0.00     2022-03-21 22:17:00+0800                        None  1.000000e+09
5   node-5  system.net.tcp.retrans_segs                      31   0.00  0.00                 0.02                 0.00     2022-03-21 22:03:00+0800                        None  2.000000e+07
6   node-3              system.io.rkb_s                      31   2.00  0.00               246.00                 0.00     2022-03-21 22:16:00+0800                        None  1.220000e+02
7   node-5           system.io.avg_q_sz                      31   0.06  0.00                 7.14                 0.00     2022-03-21 22:04:00+0800                        None  1.180000e+02
8   node-5            system.cpu.iowait                      31   0.04  0.00                 4.10                 0.00     2022-03-21 22:13:00+0800                        None  1.015000e+02
9   node-6  system.net.tcp.retrans_segs                      31   0.28  0.00                17.78                 0.00     2022-03-21 22:02:00+0800                        None  6.250000e+01
10  node-1  system.net.tcp.retrans_segs                      31   0.10  0.00                 5.37                 0.00     2022-03-21 22:06:00+0800                        None  5.270000e+01
11  node-3                system.io.r_s                      31   0.50  0.00                18.00                 0.00     2022-03-21 22:16:00+0800                        None  3.500000e+01
12  node-5               system.io.util                      31   1.40  0.00                45.85                 0.00     2022-03-21 22:14:00+0800                        None  3.175000e+01
13  node-5              system.io.await                      31   1.25  0.00                39.92                 0.00     2022-03-21 22:14:00+0800                        None  3.093600e+01
14  node-5            system.io.w_await                      31   1.25  0.00                39.92                 0.00     2022-03-21 22:14:00+0800                        None  3.093600e+01
15  node-6                system.io.r_s                      31  11.00  0.00               116.50                 0.00     2022-03-21 22:04:00+0800                        None  9.590909e+00
16  node-6              system.io.rkb_s                      31  60.00  0.00               480.00                 0.00     2022-03-21 22:04:00+0800                        None  7.000000e+00
17  node-3  system.net.tcp.retrans_segs                      31   0.08  0.00                 0.47                 0.00     2022-03-21 22:30:00+0800                        None  4.875000e+00
18  node-1            system.cpu.iowait                      31   1.42  0.06                 8.33                 0.04     2022-03-21 22:30:00+0800    2022-03-21 22:13:00+0800  4.866197e+00
19  node-3           system.io.avg_q_sz                      31   0.03  0.00                 0.09                 0.00     2022-03-21 22:25:00+0800                        None  2.000000e+00, 293)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# reuse existing metric_df and tz if present, otherwise read and set tz
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware timestamp column in Asia/Shanghai (UTC+8)
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# incident window
start = pd.Timestamp('2022-03-21 22:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 22:30:00', tz=tz)

# filter rows where kpi_name contains container.node-5 or container.node-6 and within window
mask_node5 = metric_df['kpi_name'].str.contains('container.node-5', na=False)
mask_node6 = metric_df['kpi_name'].str.contains('container.node-6', na=False)
mask_window = (metric_df['ts'] >= start) & (metric_df['ts'] <= end)

sel = metric_df[(mask_window) & (mask_node5 | mask_node6)].copy()

# helper to format timestamp to UTC+8 string
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

# aggregate per cmdb_id
grouped = sel.groupby('cmdb_id', observed=True)

def summarize_group(df):
    has5 = df['kpi_name'].str.contains('container.node-5', na=False).any()
    has6 = df['kpi_name'].str.contains('container.node-6', na=False).any()
    if has5 and has6:
        label = 'both'
    elif has5:
        label = 'node-5'
    else:
        label = 'node-6'
    total_points = int(df.shape[0])
    distinct_kpis = int(df['kpi_name'].nunique())
    earliest = fmt_ts(df['ts'].min())
    sample_kpis = df['kpi_name'].drop_duplicates().head(10).tolist()
    return pd.Series({
        'node_label': label,
        'total_points_in_window': total_points,
        'distinct_kpi_count_in_window': distinct_kpis,
        'earliest_timestamp_in_window': earliest,
        'sample_kpi_names': sample_kpis
    })

summary_df = grouped.apply(summarize_group).reset_index()

# sort by total_points_in_window desc and limit to top 20
final_df = summary_df.sort_values('total_points_in_window', ascending=False).head(20).reset_index(drop=True)

# total number of distinct cmdb_id discovered
total_cmdb_count = int(summary_df['cmdb_id'].nunique())

# display compact result
final_df, total_cmdb_count

```
Out[8]:
```
```python
summary_text = (
    "Summary of container.node-5 / container.node-6 metrics in the incident window (2022-03-21 22:00:00 to 22:30:00 UTC+8):\n\n"
    "- Total distinct cmdb_id discovered: 42.\n"
    "- Most matched components had 1,984 points in the window and 64 distinct KPI names (earliest timestamp = 2022-03-21 22:00:00+0800).\n"
    "- Node labeling: the majority of rows match 'container.node-6' (many cmdb_id labeled node-6). A smaller set match 'container.node-5' (examples in the results include adservice-2, cartservice2-0, checkoutservice-2, etc.). If a component had KPIs for both substrings it would be labeled 'both' (none of the top entries showed 'both').\n"
    "- Typical KPI names observed (sample): container.node-6.container_cpu_cfs_periods, container.node-6.container_memory_failures.*, container.node-6.container_network_receive_MB.eth0, container.node-6.container_network_transmit_MB.eth0, and similar container.node-5 variants.\n\n"
    "Interpretation:\n"
    "- Many container-level metrics in the window are tied to node-6 (dominant), suggesting heavy container telemetry coming from node-6 during the incident window. The uniform earliest timestamp (22:00) shows the metrics are present from the start of the window.\n"
    "- To investigate, inspect the sampled KPI names for node-6 and the few node-5 entries for spikes in CPU, memory failures, network receive/transmit metrics and correlate with node-level alerts (I/O and retransmissions found earlier).\n"
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id node_label  total_points_in_window  distinct_kpi_count_in_window earliest_timestamp_in_window                                   sample_kpi_names
0          adservice-0     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
1          adservice-1     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
2          adservice-2     node-5                    1984                            64     2022-03-21 22:00:00+0800  [container.node-5.container_cpu_cfs_periods, c...
3         adservice2-0     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
4        cartservice-0     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
5        cartservice-1     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
6        cartservice-2     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
7       cartservice2-0     node-5                    1984                            64     2022-03-21 22:00:00+0800  [container.node-5.container_cpu_cfs_periods, c...
8    checkoutservice-0     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
9    checkoutservice-1     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
10   checkoutservice-2     node-5                    1984                            64     2022-03-21 22:00:00+0800  [container.node-5.container_cpu_cfs_periods, c...
11  checkoutservice2-0     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
12   currencyservice-0     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
13   currencyservice-1     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
14   currencyservice-2     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
15  currencyservice2-0     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
16      emailservice-0     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
17      emailservice-1     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
18      emailservice-2     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c...
19     emailservice2-0     node-6                    1984                            64     2022-03-21 22:00:00+0800  [container.node-6.container_cpu_cfs_periods, c..., 42)```
```